# Prepare Data for Qwen2.5-7B Finetuning

This notebook converts the synthesized ToM questions into the format needed for finetuning.

In [1]:
import pandas as pd
import json
import glob
from pathlib import Path

print("Libraries imported successfully")

Libraries imported successfully


## Step 1: Load and Concatenate Synthesized Data

In [11]:
# Load all CSV files from synthesized data directory
synthesized_data_dir = "../results/synthesized_data/data"
csv_files = glob.glob(f"{synthesized_data_dir}/*.csv")

print(f"Found {len(csv_files)} CSV files in {synthesized_data_dir}:")
for f in csv_files:
    print(f"  - {Path(f).name}")

# Load and concatenate all CSV files
synthesized_dfs = []
for csv_file in csv_files:
    df = pd.read_csv(csv_file)
    synthesized_dfs.append(df)
    print(f"Loaded {Path(csv_file).name}: {len(df)} rows")

if synthesized_dfs:
    synthesized_df = pd.concat(synthesized_dfs, ignore_index=True)
    print(f"\n✓ Concatenated {len(synthesized_dfs)} files: {len(synthesized_df)} total rows")
    print(f"Columns: {list(synthesized_df.columns)}")
else:
    raise ValueError(f"No CSV files found in {synthesized_data_dir}")

Found 7 CSV files in ../results/synthesized_data/data:
  - cluster4_80questions.csv
  - cluster45_30questions.csv
  - cluster5_10Q.csv
  - cluster5_11Qs.csv
  - cluster5_20Q.csv
  - cluster5_30Q.csv
  - cluster9_10q.csv
Loaded cluster4_80questions.csv: 80 rows
Loaded cluster45_30questions.csv: 30 rows
Loaded cluster5_10Q.csv: 10 rows
Loaded cluster5_11Qs.csv: 11 rows
Loaded cluster5_20Q.csv: 20 rows
Loaded cluster5_30Q.csv: 30 rows
Loaded cluster9_10q.csv: 10 rows

✓ Concatenated 7 files: 191 total rows
Columns: ['cluster_id', 'question_index', 'story', 'question', 'option_a', 'option_b', 'option_c', 'option_d', 'correct_answer', 'cluster_summary']


## Step 2: Add TASK Column and Standardize Column Names

In [12]:
# Add TASK column based on cluster_id
# cluster_id 3 = table cluster 4, cluster_id 4 = table cluster 5
synthesized_df['TASK'] = synthesized_df['cluster_id'].apply(
    lambda x: f"synthesize data for cluster {x+1}"
)

print(f"✓ Added TASK column")
print(f"Task distribution:")
print(synthesized_df['TASK'].value_counts())

# Standardize column names to match train.csv format (uppercase with hyphens)
column_mapping = {
    'story': 'STORY',
    'question': 'QUESTION',
    'option_a': 'OPTION-A',
    'option_b': 'OPTION-B',
    'option_c': 'OPTION-C',
    'option_d': 'OPTION-D',
    'correct_answer': 'ANSWER'
}

synthesized_df = synthesized_df.rename(columns=column_mapping)

print(f"\n✓ Standardized column names")
print(f"Columns after standardization: {list(synthesized_df.columns)}")

✓ Added TASK column
Task distribution:
TASK
synthesize data for cluster 4    100
synthesize data for cluster 5     81
synthesize data for cluster 9     10
Name: count, dtype: int64

✓ Standardized column names
Columns after standardization: ['cluster_id', 'question_index', 'STORY', 'QUESTION', 'OPTION-A', 'OPTION-B', 'OPTION-C', 'OPTION-D', 'ANSWER', 'cluster_summary', 'TASK']


## Step 3: Load Original Train Data

In [13]:
# Load train.csv
train_csv_path = "../train.csv"
train_df = pd.read_csv(train_csv_path)

## Step 4: Concatenate Datasets (Shared Columns Only)

In [14]:
# Find shared columns (case-insensitive comparison)
train_cols_lower = {col.lower(): col for col in train_df.columns}
synth_cols_lower = {col.lower(): col for col in synthesized_df.columns}

# Get shared column names (using train.csv naming convention)
shared_cols = []
for col_lower in train_cols_lower.keys():
    if col_lower in synth_cols_lower:
        train_col = train_cols_lower[col_lower]
        synth_col = synth_cols_lower[col_lower]
        shared_cols.append(train_col)
        
        # Rename synthesized column to match train.csv if different
        if synth_col != train_col:
            synthesized_df = synthesized_df.rename(columns={synth_col: train_col})

print(f"✓ Found {len(shared_cols)} shared columns:")
print(f"  {shared_cols}")

# Select only shared columns from both dataframes
train_df_subset = train_df[shared_cols]
synthesized_df_subset = synthesized_df[shared_cols]

# Concatenate
combined_df = pd.concat([train_df_subset, synthesized_df_subset], ignore_index=True)

print(f"\n✓ Combined dataset:")
print(f"  Train data: {len(train_df_subset)} rows")
print(f"  Synthesized data: {len(synthesized_df_subset)} rows")
print(f"  Total: {len(combined_df)} rows")
print(f"  Columns: {list(combined_df.columns)}")

✓ Found 8 shared columns:
  ['STORY', 'QUESTION', 'OPTION-A', 'OPTION-B', 'OPTION-C', 'OPTION-D', 'ANSWER', 'TASK']

✓ Combined dataset:
  Train data: 2003 rows
  Synthesized data: 191 rows
  Total: 2194 rows
  Columns: ['STORY', 'QUESTION', 'OPTION-A', 'OPTION-B', 'OPTION-C', 'OPTION-D', 'ANSWER', 'TASK']


In [15]:
# Modify ANSWER column: wrap each answer in double brackets
# Example: "B" -> "[[B]]"
# combined_df['ANSWER'] = combined_df['ANSWER'].apply(lambda x: f"[[{x}]]")
combined_df = combined_df.sample(frac=1).reset_index(drop=True)

In [16]:
combined_df.to_csv("finetune_data.csv", index=False)

## Step 5

In [17]:
def create_training_sample(row):
    """
    Convert a single row to Qwen2.5 training format.
    Matches the format from ToM/prompt-noCoT.txt
    """
    # Format the question following prompt-noCoT.txt structure
    user_message = f"""Below is a multiple-choice question with a story and serveral answer options. Based on the content of the story and the given question, please infer the most likely answer and output the answer index.

Note:
(1) Please only output the most likely answer index in the format: [[Answer Index]], for example, if the most likely answer option is 'A. Handbag', then output '[[A]]';
(2) You must choose one of the given answer options 'A, B, C, D' as the most likely answer, regardless of whether the story provides enough information. If you think there is not enough information in the story to choose an answer, please output the most likely answer among "[[A]]", "[[B]]", "[[C]]", or "[[D]]" based on the current story;
(3) Please only output the most likely answer index based on the given information, and do not output any other content.

[Story]
{row['STORY']}

[Question]
{row['QUESTION']}

[Candidate Answers]
A. {row['OPTION-A']} B. {row['OPTION-B']} C. {row['OPTION-C']} D. {row['OPTION-D']}

Output the most likely answer among "[[A]]", "[[B]]", "[[C]]", or "[[D]]", and nothing else. Do not think."""
    
    # The correct answer (already wrapped in [[]] from previous step)
    assistant_message = row['ANSWER']
    
    return {
        "messages": [
            {"role": "user", "content": user_message},
            {"role": "assistant", "content": assistant_message}
        ]
    }

# Convert all rows
training_data = [create_training_sample(row) for _, row in combined_df.iterrows()]

print(f"✓ Created {len(training_data)} training samples")
print(f"\nExample sample:")
print(json.dumps(training_data[0], indent=2, ensure_ascii=False))

✓ Created 2194 training samples

Example sample:
{
  "messages": [
    {
      "role": "user",
      "content": "Below is a multiple-choice question with a story and serveral answer options. Based on the content of the story and the given question, please infer the most likely answer and output the answer index.\n\nNote:\n(1) Please only output the most likely answer index in the format: [[Answer Index]], for example, if the most likely answer option is 'A. Handbag', then output '[[A]]';\n(2) You must choose one of the given answer options 'A, B, C, D' as the most likely answer, regardless of whether the story provides enough information. If you think there is not enough information in the story to choose an answer, please output the most likely answer among \"[[A]]\", \"[[B]]\", \"[[C]]\", or \"[[D]]\" based on the current story;\n(3) Please only output the most likely answer index based on the given information, and do not output any other content.\n\n[Story]\nDongdong and Xiaoning a

## Step 6: Split into Train/Validation

In [19]:
from sklearn.model_selection import train_test_split

# Split 90% train, 10% validation
train_data, val_data = train_test_split(training_data, test_size=0.1, random_state=42)

print(f"✓ Split complete:")
print(f"  Train samples: {len(train_data)}")
print(f"  Validation samples: {len(val_data)}")

✓ Split complete:
  Train samples: 1974
  Validation samples: 220


In [21]:
output_dir = Path("../finetune")
output_dir.mkdir(exist_ok=True)

# Save train data
train_path = output_dir / "train.jsonl"
with open(train_path, 'w', encoding='utf-8') as f:
    for sample in train_data:
        f.write(json.dumps(sample, ensure_ascii=False) + '\n')

# Save validation data
val_path = output_dir / "val.jsonl"
with open(val_path, 'w', encoding='utf-8') as f:
    for sample in val_data:
        f.write(json.dumps(sample, ensure_ascii=False) + '\n')

print(f"✓ Saved training data to {train_path}")
print(f"✓ Saved validation data to {val_path}")

✓ Saved training data to ../finetune/train.jsonl
✓ Saved validation data to ../finetune/val.jsonl


In [22]:
# Read back and verify
with open(train_path, 'r', encoding='utf-8') as f:
    first_sample = json.loads(f.readline())

print("Sample training data:")
print(json.dumps(first_sample, indent=2, ensure_ascii=False))

# Check data statistics
import numpy as np

user_lengths = []
assistant_lengths = []

for sample in train_data:
    user_lengths.append(len(sample['messages'][0]['content']))
    assistant_lengths.append(len(sample['messages'][1]['content']))

print(f"\nData Statistics:")
print(f"User message length - Mean: {np.mean(user_lengths):.0f}, Max: {np.max(user_lengths)}")
print(f"Assistant message length - Mean: {np.mean(assistant_lengths):.0f}, Max: {np.max(assistant_lengths)}")

print(f"\n✓ Data preparation complete!")
print(f"  Total samples: {len(training_data)}")
print(f"  Train: {len(train_data)} | Val: {len(val_data)}")
print(f"  Ready for finetuning!")

Sample training data:
{
  "messages": [
    {
      "role": "user",
      "content": "Below is a multiple-choice question with a story and serveral answer options. Based on the content of the story and the given question, please infer the most likely answer and output the answer index.\n\nNote:\n(1) Please only output the most likely answer index in the format: [[Answer Index]], for example, if the most likely answer option is 'A. Handbag', then output '[[A]]';\n(2) You must choose one of the given answer options 'A, B, C, D' as the most likely answer, regardless of whether the story provides enough information. If you think there is not enough information in the story to choose an answer, please output the most likely answer among \"[[A]]\", \"[[B]]\", \"[[C]]\", or \"[[D]]\" based on the current story;\n(3) Please only output the most likely answer index based on the given information, and do not output any other content.\n\n[Story]\nIn the busy morning, the bus is as lively as usual